# 04. Temporal Household Behavior

## Mục tiêu

> Hiểu hành vi household theo thời gian: mức độ hoạt động, độ dài lịch sử và khoảng quay lại.

Notebook dùng `DAY` và `WEEK_NO` là ordinal time. Mọi kết luận hiện dựa trên DAY 1–711, chưa gắn được ngày lịch thực.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl

from propensity.eda import (
    build_household_interpurchase_features,
    build_household_behavior_shift,
    build_household_lifecycle,
    build_household_rolling_windows,
    profile_return_intervals,
    profile_historical_activity_coverage,
    summarize_household_distributions,
    summarize_period_activity,
)
from propensity.eda.dunnhumby import DUNNHUMBY_TRANSACTION_PROFILE_CONFIG
from propensity.eda.visualization import (
    plot_household_interpurchase_dynamics,
    plot_household_behavior_change,
    plot_historical_activity_coverage,
    plot_household_behavior_shift,
    plot_last_purchase_recency,
    plot_lifecycle_coverage_exposure,
    plot_return_interval_distribution,
)
from propensity.io import scan_csv
from propensity.schema.dunnhumby import TRANSACTION_SCHEMA_OVERRIDES

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)

## 1. Load transaction data

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

transactions = scan_csv(
    project_root / "data" / "raw" / "Dunnhumby" / "transaction_data.csv",
    schema_overrides=TRANSACTION_SCHEMA_OVERRIDES,
)
config = DUNNHUMBY_TRANSACTION_PROFILE_CONFIG

## 2. Tạo temporal evidence tables

In [ ]:
daily_activity = summarize_period_activity(transactions, config.time_col, config)
weekly_activity = summarize_period_activity(
    transactions, config.period_col, config
)
household_distributions = summarize_household_distributions(
    transactions, config
)
household_lifecycle = build_household_lifecycle(transactions, config)
household_interpurchase = build_household_interpurchase_features(
    transactions, config
)
household_windows = build_household_rolling_windows(
    transactions, config, window_weeks=13, step_weeks=4
)
activity_coverage = profile_historical_activity_coverage(
    transactions,
    config,
    windows_days=(3, 7, 14, 21, 30, 60, 90, 180, 365),
)
household_behavior_shift = build_household_behavior_shift(
    transactions, config, recent_days=30, history_days=365
)
return_intervals = profile_return_intervals(transactions, config)

## 3. Entry cohort và exposure bias

Panel trái trả lời household gia nhập từ đầu hay rải rác, đồng thời làm nổi bật các cohort xuất hiện trong 30/60/90 ngày cuối. Panel phải kiểm tra raw basket count có tăng cơ học theo số ngày được quan sát hay không. `Exposure days` được tính từ lần mua đầu tiên đến snapshot, bao gồm cả hai đầu.

In [ ]:
lifecycle_figure = plot_lifecycle_coverage_exposure(household_lifecycle)
lifecycle_figure

### Cách đọc

- Household gần như đều gia nhập trong 20 tuần đầu; cohort lớn nhất là tuần 16 với 197 household. Chỉ 1/2.500 household gia nhập trong 60–90 ngày cuối và không có household nào gia nhập trong 30 ngày cuối.
- Spearman giữa exposure và basket count chỉ bằng 0,06; đường median cũng gần như nằm ngang trong vùng exposure chính. Raw basket count của population hiện tại vì vậy không bị exposure chi phối mạnh.
- Các household exposure rất ngắn là ngoại lệ hiếm. Vẫn nên giữ exposure-normalized frequency cho scoring tại snapshot tương lai, khi household mới có thể xuất hiện nhiều hơn; correlation không chứng minh quan hệ nhân quả.

## 4. Last purchase và recency

Panel trái cho biết household tập trung mua lần cuối ở giai đoạn nào. Panel phải chuyển cùng thông tin sang `recency_days = snapshot_day - last_purchase_day`, giúp đọc trực tiếp tỷ lệ household còn hoạt động trong 7/14/30/60/90 ngày gần snapshot.

In [ ]:
last_purchase_figure = plot_last_purchase_recency(household_lifecycle)
last_purchase_figure

### Kết luận last purchase

- Median recency là 6 ngày; 56,3% household mua trong 7 ngày và 68,3% mua trong 14 ngày trước snapshot.
- 81,8% household mua trong 30 ngày, 89,8% trong 60 ngày và 93,2% trong 90 ngày. Population nhìn chung vẫn hoạt động gần cuối observation window.
- Recency có đuôi dài đến 657 ngày. Đây là thời gian im lặng quan sát được, chưa phải churn probability; cần đọc cùng nhịp mua điển hình của từng household.

## 5. Activity theo ngày và tuần

Hai bảng giữ số product lines, baskets, households, products, sales, quantity và discount theo từng period. Đây là nền tảng để phân tích continuity, trend và seasonality.

In [ ]:
display(daily_activity.select(pl.exclude("period")).describe())
display(weekly_activity.select(pl.exclude("period")).describe())

### Kết luận period activity

- `daily_activity` có 711 dòng và `weekly_activity` có 102 dòng, đúng với độ phủ DAY/WEEK đã kiểm tra.
- Mỗi period chứa các measure ở cùng grain, cho phép xác định giai đoạn volume thấp/cao trước khi phân tích trend hoặc seasonality.
- `total_discount` là độ lớn khoản giảm, được tạo bằng cách đổi dấu tổng ba discount columns; quy ước dấu vẫn cần xác nhận từ data dictionary.

## 6. Active periods và độ dài lịch sử household

`n_active_days` đếm số DAY distinct có mua hàng. `span_days` tính từ DAY mua đầu tiên đến DAY mua cuối cùng, bao gồm cả hai đầu.

In [ ]:
household_distributions.filter(
    pl.col("metric").is_in(["n_active_days", "span_days"])
)

### Kết luận active history

- Median household có 70 active days.
- Median `span_days` là 627; p25 = 600 và p90 = 688.
- Phần lớn household có lịch sử đủ dài cho observation window 12 tháng, nhưng độ dài lịch sử không đồng nghĩa hoạt động liên tục. Continuity cần đọc cùng `n_active_days` và purchase intervals.

## 7. Khoảng thời gian quay lại

Khoảng quay lại là chênh lệch `DAY` giữa hai active days distinct liên tiếp của cùng household. Nhiều baskets trong cùng ngày không tạo gap bằng 0.

In [ ]:
return_interval_figure = plot_return_interval_distribution(return_intervals)
return_interval_figure

household_interval_figure = plot_household_interpurchase_dynamics(
    household_interpurchase, return_intervals
)
household_interval_figure

### Kết luận khoảng quay lại

- Có 223.033 khoảng quay lại hợp lệ từ 2.500 households.
- 24,0% lượt quay lại sau 1 ngày; cumulative share đạt 52,1% trong 3 ngày, 77,9% trong 7 ngày và 90,7% trong 14 ngày.
- Median = 3 ngày, p90 = 14, p95 = 22, p99 = 61 và max = 666 ngày.
- Phân phối chưa xử lý right censoring tại DAY 711 và không phải xác suất churn.
- Khi mỗi household chỉ đóng góp một giá trị, median của `median_interval_days` tăng lên 6 ngày, gấp đôi median event-level 3 ngày. Điều này xác nhận các lượt quay lại toàn hệ thống bị nghiêng về nhóm mua dày hơn household điển hình.
- Trong panel lateness, `1×` là median interval của chính household, không phải mean/median toàn population. Trong 2.479 household có ít nhất 3 intervals: 47,5% chưa đến nhịp riêng, 22,3% trễ 1–2 lần, 14,6% trễ 2–4 lần và 15,5% trễ trên 4 lần median interval.
- Panel CV cho thấy mật độ chính nằm quanh CV 0,7–1,3: nhiều household có nhịp mua biến động đáng kể thay vì một chu kỳ cố định. Các lateness bands chỉ là mô tả; chưa thể vẽ churn rate khi chưa có nhãn outcome.

## 8. Historical activity coverage

So sánh các trailing windows 3/7/14/21/30/60/90/180/365 ngày trên cùng tập household tại cutoff DAY 711. Household không mua trong window không bị loại mà được giữ với zero baskets. `Gap-ready` yêu cầu ít nhất hai active days distinct; continuity tối thiểu yêu cầu hoạt động trong ít nhất hai tuần distinct.

In [ ]:
activity_coverage_figure = plot_historical_activity_coverage(
    activity_coverage
)
activity_coverage_figure

### Kết luận coverage

- Với 3 ngày, chỉ 32,9% household có basket, median bằng 0 và 7,3% đủ tính purchase gap. Với 7 ngày, coverage tăng lên 52,9% nhưng chỉ 25,1% gap-ready. Hai window này phù hợp cho tín hiệu recency/hotness, không phù hợp để ước lượng cadence.
- Với 14 ngày, 66,9% có hoạt động và 45,2% gap-ready; đến 21 ngày các tỷ lệ tương ứng là 76,1% và 57,5%. Median basket count mới chỉ đạt 1 và 2, nên regularity vẫn thiếu evidence.
- Với 30 ngày, 81,6% household có hoạt động nhưng chỉ 38,2% có ít nhất 5 baskets; 67,6% đủ hai active days để tính một purchase gap. Median chỉ có 3 baskets, nên window này phù hợp cho recency hơn regularity.
- Với 90 ngày, 93,0% có hoạt động, 87,1% gap-ready, 86,5% hoạt động trong ít nhất hai tuần và median có 10 baskets. Đây là điểm cân bằng hợp lý cho cadence features cơ bản, dù chỉ 73,1% có ít nhất 5 baskets.
- Với 180 ngày, 85,9% có ít nhất 5 baskets và 93,9% gap-ready; phù hợp hơn khi cần CV hoặc trend ổn định. Median basket count luôn được tính trên cả 2.500 household, bao gồm zero; không có một T tối ưu cho mọi metric.

## 9. Long-term baseline → recent behavior shift

So sánh recent 30 ngày với 365 ngày lịch sử ngay trước đó, không chồng lấn. Frequency transition được tính bằng recent baskets so với historical baskets chuẩn hóa về 30 ngày: `<0,8×` là giảm nhịp, `0,8–1,25×` là ổn định và `>1,25×` là tăng tốc; zero recent baskets được giữ thành Dormant. Median products per basket là chiều hành vi riêng, tránh gộp frequency và basket size thành một score khó giải thích.

In [ ]:
behavior_shift_figure = plot_household_behavior_shift(
    household_behavior_shift
)
behavior_shift_figure

### Cách đọc behavior shift

- Trên toàn bộ 2.500 household: 29,0% tăng tốc, 25,2% ổn định, 27,0% giảm nhịp, 18,4% không mua trong recent 30 ngày và 0,2% mới hoạt động. Scatter hiển thị phần trung tâm đến joint p99; transition bars vẫn giữ toàn population.
- Median products/basket đều tăng ở ba nhóm còn active: tăng tốc từ 5 lên 5,5; ổn định từ 6 lên 7; giảm nhịp từ 5 lên 7,5. Vì vậy giảm frequency không đồng nghĩa basket shrinking—nhóm giảm nhịp có thể đang gom mua thành ít chuyến hơn nhưng giỏ lớn hơn.
- Transition bars là lớp nén để truyền thông, không thay thế dữ liệu gốc. Slopegraph là đại diện nhóm, không phải quỹ đạo của từng household; Dormant được giữ trong population nhưng không có recent basket size để đưa vào slopegraph.

## 10. Thay đổi hành vi theo thời gian

Rolling window 13 tuần, cập nhật mỗi 4 tuần, phủ toàn bộ lịch sử bằng các cửa sổ chồng lấn. Baseline là window đầu tiên đạt ít nhất 90% số active household tối đa và 90% mature basket rate, tránh household đã xuất hiện nhưng chưa đủ exposure làm méo chỉ số. Hai panel dưới dùng hai giai đoạn 13 tuần không chồng lấn gần snapshot để đo thay đổi cadence ở household-level.

In [ ]:
behavior_change_figure = plot_household_behavior_change(household_windows)
behavior_change_figure

### Kết luận behavior change

- Sau baseline tuần 25, median basket rate cao hơn khoảng 10%, trong khi median interval thường cao hơn 10–20%; median CV chủ yếu dao động quanh 96–100. Basket volume tăng nhưng khoảng giữa các active days không ngắn lại, có thể phản ánh nhiều basket hơn trong cùng ngày mua.
- So sánh tuần 77–89 với 90–102 trên 1.864 household đủ evidence: 27,9% mua nhanh hơn, 38,2% tương đối ổn định, 21,2% chậm lại và 12,7% chậm lại mạnh.
- Hexbin cho thấy household nằm phía trên đường chéo mua chậm lại, phía dưới mua nhanh hơn. Các change bands là mô tả theo tỷ lệ recent/prior median interval, chưa phải churn states; 636 household thiếu ít nhất 3 intervals ở một trong hai window nên không được phân loại.

## 11. Phần temporal tiếp theo

Bảng household-level hiện đã có first/last purchase, exposure và basket count. Các bước tiếp theo sẽ bổ sung recency, regularity và continuity trước khi phân tích trend và seasonality. Mọi feature phải chỉ dùng dữ liệu có `DAY <= snapshot` để tránh leakage.